In [58]:
# MIT License
# 
# Copyright (c) 2024 [저작권자 이름]
# 
# This software is released under the MIT License.
# https://opensource.org/licenses/MIT

In [59]:
from pyhwpx import Hwp
import win32com.client as win32
import pandas as pd
hwp = Hwp()
hwp.Open(r"../template2.hwp") # 주간행사계획 한글파일 상대경로

True

time 컬럼 초기화 하는 방법

In [60]:
#이건 무조건 젤처음으로 실행해야됨


#처음  time column  필드값 지워놓기
def time_column_initail(date_count): 
    hwp.set_pos(15,0,0)
    hwp.TableCellBlock() #그냥 테이블 셀 선택
    hwp.TableCellBlockExtend() #선택된셀에서 f5누른거랑 같은 기능
    hwp.TableColPageDown()
    hwp.set_cur_field_name("")   
    for i in range (date_count):
        hwp.set_pos(15+i*13,0,0)
        hwp.set_cur_field_name("time")
        print(hwp.get_pos())

#근데 같은 월일안에서 TableLowerCell()을 실행하면 list값이 6씩증가하고 
# TableLowerCell()을 실행했을때 다른 월일 row로 넘어가면 +7이 된다


time_column_initail(7)


(15, 0, 0)
(28, 0, 0)
(41, 0, 0)
(54, 0, 0)
(67, 0, 0)
(80, 0, 0)
(93, 0, 0)


해결해야할 문제 3 -> 완료
한컬럼을 전부 date필드로 정한다음 row를 추가하는 함수에서 index를 +1만큼씩 더 가기
vs 
그냥 setpos로 해당 위치에 있는 셀에 필드값 설정하기

한컬럼 전부를 time필드로 만들고 rowappend하면 안됨 -> row가 바뀔때마다 time field인덱스값도 변경됨

In [61]:
# #데이터 전처리
# df = pd.read_excel(r"../eventdummy.xlsx")
# #엑셀에서 더미데이터 read
# df = df.replace(r'\n', '', regex=True)
# # 데이터안에 \n 다지우기
# 
# df = df.fillna(" ")  # NaN 값을 으로 채우기
# df.tail()

import pandas as pd
import re

#데이터 전처리
df = pd.read_excel(r"../eventdummy.xlsx")
#엑셀에서 더미데이터 read
df = df.replace(r'\n', '', regex=True)
# 데이터안에 \n 다지우기
#가장 마지막 row의 year값 가져오기
date_df = df['date']
start_date = date_df.iloc[-1]
# 첫 번째 점(.) 앞의 숫자만 추출
year = "20"+start_date.split(".")[0]
# 출력
print(f"추출된 년도: {year}")


df = df.fillna(" ")  # NaN 값을 " "으로 채우기
# df['date'] = df['date'].apply(lambda x: re.sub(r'^\d{2}\.', '', x))

df['date'] = df['date'].str.split('.',n=1).str[1] #년도 제거 코드

#처음과 마지막 date 뽑아오기

# 입력 날짜 문자열

df.tail()

추출된 년도: 2024


,date,time,eventname,place,personnel,department,note
14,4.13 (토),10:00:00,·2024년 특성화 사업 제1회차‘인문학 콘서트’,유성문화원 1층전시실,30.0,문화관광과,
15,4.13 (토),10:00:00,·행복한 문화학교 「햄스터 로봇 코딩」,유성도서관,14.0,도서관운영과,
16,4.13 (토),15:00:00,·노은도서관 동화 들려주기(Story Time),노은도서관,40.0,도서관운영과,
17,4.13 (토),15:00:00,·독서프로그램 「귀 기울여 영어동화」 영어원서 읽기,어린이영어마을도서관,10.0,도서관운영과,
18,4.14 (일),,,,,,


In [62]:
#처음 날짜 바꾸는 코드

date_df = df["date"]
date_df.iloc[0]


def split_date_components(date_str):
    # 4.8 (월) -> 4. 8. 형식으로 교체하는 함수
    # 정규식을 사용하여 숫자와 괄호 부분을 추출하고, 원하는 형식으로 변경
    result = re.sub(r"(\d+)\.(\d+)\s\([^)]+\)", r"\1. \2.", date_str)

    print(result)

    return result



# 첫 번째와 마지막 날짜에서 월, 일, 요일을 분리하는 함수
def get_first_last_date_components(date_df):
    # df의 첫 번째와 마지막 인덱스에서 date 값 추출
    first_date = date_df.iloc[0] # 첫 번째 인덱스의 date 값
    last_date = date_df.iloc[-1]  # 마지막 인덱스의 date 값
    
    # 첫 번째와 마지막 날짜의 월, 일, 요일을 분리
    first_date_components = split_date_components(first_date)
    last_date_components = split_date_components(last_date)
    
    return first_date_components, last_date_components


first_date_components,last_date_components= get_first_last_date_components(date_df)




4. 8.
4. 14.


In [63]:
date_counts = df['date'].value_counts()
date_counts

date
4.13 (토)    6
4.11 (목)    4
4.8 (월)     3
4.12 (금)    2
4.9 (화)     2
4.10 (수)    1
4.14 (일)    1
Name: count, dtype: int64

In [70]:
import re
#위에처럼 count순으로 sort된다 
#string 날짜데이터를 날짜순으로 sort하는 코드
date_counts = df['date'].value_counts().reset_index()
date_counts.columns = ['date', 'count']  # 열 이름 지정

# 월과 일만 추출하는 함수 정의
def extract_month_day(date_str):
    match = re.search(r'(\d+)\.\s*(\d+)', date_str)
    if match:
        return int(match.group(1)), int(match.group(2))  # 월, 일 반환
    return None

# 'month', 'day' 열 추가
date_counts[['month', 'day']] = date_counts['date'].apply(lambda x: pd.Series(extract_month_day(x)))

# 월과 일 기준으로 정렬
date_counts = date_counts.sort_values(by=['month', 'day']).drop(columns=['month', 'day']).reset_index(drop=True)

#date 컬럼만
date_column_df = date_counts[['date']]

# 정렬된 결과 출력
print(date_counts)
print(date_column_df)

       date  count
0   4.8 (월)      3
1   4.9 (화)      2
2  4.10 (수)      1
3  4.11 (목)      4
4  4.12 (금)      2
5  4.13 (토)      6
6  4.14 (일)      1
       date
0   4.8 (월)
1   4.9 (화)
2  4.10 (수)
3  4.11 (목)
4  4.12 (금)
5  4.13 (토)
6  4.14 (일)


#데이터 갯수만큼 열 만듦 
위에 코드 실행하면
date 마다 데이터 갯수가 뜨는데
해당 데이터 갯수만큼 열을 나눠주는 코드 

In [65]:
# print(count_value)


for index, row in date_counts.iterrows():
    pset = hwp.HParameterSet.HTableDeleteLine
    hwp.move_to_field(f'time{{{{{index}}}}}')
    print(index)
    count = row['count']
    
    if count>2 :
        for i in range(count - 2):
            hwp.TableAppendRow()
            print(f"{index}" + ": append" )
    elif (count == 2):
        continue
    else :
        hwp.TableLowerCell()
        hwp.HAction.GetDefault("TableDeleteRow", pset.HSet)
        #TLqkf
        hwp.HAction.Execute("TableDeleteRow", pset.HSet)
        print(f"{index}"+": delete" )
        
        
    

0
0: append
1
2
2: delete
3
3: append
3: append
4
5
5: append
5: append
5: append
5: append
6
6: delete


date컬럼에 date값들 집어넣는 코드    (주의사항 : 한글의 월일 column에 셀 필드값이 date로 되어야함)


In [66]:
# date 데이터 넣기
for index, row in date_column_df.iterrows():
    hwp.move_to_field(f'date{{{{{index}}}}}')
    hwp.insert_text(row['date'])
    print(row['date'])

4.8 (월)
4.9 (화)
4.10 (수)
4.11 (목)
4.12 (금)
4.13 (토)
4.14 (일)


date제외 데이터를 전부 삽입함

In [67]:
#date 제외 데이터 넣기
df_no_date = df.drop(columns=['date'])
hwp.set_pos(15,0,0)
# 셀 전부 선택하는 코드 ->
hwp.TableCellBlock()
hwp.TableCellBlockExtend()
hwp.TableColEnd()
hwp.TableColPageDown()
# 선택된 셀을 imsi필드로 바꾸고 
hwp.set_cur_field_name("imsi")

hwp.put_field_text("imsi", df_no_date.values.flatten().tolist())


In [68]:
import os 

download = str(os.path.join(os.path.expanduser("~"), 'downloads')) + "\\output.hwp"
hwp.SaveAs(download)
# 세이브하는 코드

True

In [69]:
# 적었던 필요없는 코드들


#hwp.MoveToField("date", True, True, False)
#-> 왜인지 모르겠지만 첫번쨰 date로 가진다 ㅇㅇ
#hwp.TableSplitCell()
# 표 나누기
# =========--------------
# times = (df['time'])
# for index,time in enumerate(times):
#     print(index , time)
#     hwp.put_field_text("time",f"{time}",index)

        # for i in range(7):
#     
#     hwp.move_to_field(f'time{{{{{i}}}}}')
#     hwp.insert_text(f"{i}")
        
  #------------------------------------------#      
# hwp.get_pos() # 위치
# 
# def cell_initialize():
#     hwp.set_pos(15,0,0) # 첫번째 시작셀
#     hwp.set_cur_field_name("time")
#     for i in range()
#         